# Rayong MARCO — Template 3 (ตัวเลขลายมือ)
**Pipeline:** รูปถ่าย → จัดภาพ (marker 4 จุด) → **YOLO** หาแถวคำตอบ/ช่องรหัส → แบ่งแถวเป็น 10 ช่อง → Character Check (ว่าง/มีตัวเลข) → **CNN ที่ propose เอง** อ่านตัวเลข 0–9 → JSON → ให้คะแนน

| ส่วน | ทำอะไร |
|---|---|
| 0 | ติดตั้ง + เชื่อม Google Drive |
| 1 | ฟังก์ชันหลัก (`t3core.py`) |
| 2 | วัด layout จาก template เปล่า (ทำครั้งเดียว) |
| 3 | ทดสอบจัดภาพกับรูปถ่ายจริง 1 รูป |
| 4 | สร้าง dataset ตัวเลข (สำหรับ CNN) |
| 5 | สร้าง dataset YOLO (label อัตโนมัติ) + เทรน YOLO |
| 6 | เทรน CNN (11 คลาส: 0–9 + blank) |
| 7 | ตรวจข้อสอบจริง → JSON → คะแนน + CAR/WAR |

> เปิด GPU ก่อน: **Runtime → Change runtime type → T4 GPU**

## 0. ติดตั้ง + เชื่อม Drive

ใช้โฟลเดอร์ที่มีอยู่แล้ว:
```
MyDrive/
├── Raw_Dataset/          ← มีอยู่แล้ว: 0/ … 9/ รูปถ่ายแผ่นที่ทุกช่องเขียนเลขเดียวกัน
│   └── blank/            ← (ไม่บังคับ) รูปแผ่นเปล่า ช่วยสอนคลาส blank
└── CV_T3/                ← notebook สร้างให้เองตอนรันเซลล์นี้ (ไม่ต้องสร้างเอง)
    ├── eval/             ← แผ่นทดสอบจริง IMG_xxx.jpg + IMG_xxx.csv (เฉลย) — ใส่ทีหลัง
    └── out/              ← ผลลัพธ์ทั้งหมด (cells, โมเดล, json)

template เปล่า 3 ไฟล์อ่านจาก `Raw_Dataset/blank/` (ไฟล์ที่ชื่อมี page1 / page2 / page3)
```
ถ้า Raw_Dataset ไม่ได้อยู่ใน My Drive ตรง ๆ ให้แก้ `RAW_DIR` ในเซลล์ล่าง

In [ ]:
!pip -q install ultralytics pillow-heif
from google.colab import drive
drive.mount('/content/drive')

import os, glob
from collections import Counter
RAW_DIR = '/content/drive/MyDrive/Raw_Dataset'     # โฟลเดอร์ dataset ที่มีอยู่แล้ว
BASE    = '/content/drive/MyDrive/CV_T3'           # eval / out (สร้างให้อัตโนมัติ)
TEMPLATE_DIR = f'{RAW_DIR}/blank'                  # template เปล่า page1/2/3
for d in ['eval', 'out']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# สำรวจ Raw_Dataset: จำนวนไฟล์และนามสกุลในแต่ละโฟลเดอร์
assert os.path.isdir(RAW_DIR), f'ไม่พบ {RAW_DIR} — แก้ RAW_DIR ให้ตรง'
for lab in sorted(os.listdir(RAW_DIR)):
    fs = [f for f in glob.glob(f'{RAW_DIR}/{lab}/**/*', recursive=True) if os.path.isfile(f)]
    print(f'{lab:>6}: {len(fs):4d} ไฟล์', dict(Counter(os.path.splitext(f)[1].lower() for f in fs)))

## 1. ฟังก์ชันหลัก
เขียนลงไฟล์ `t3core.py` แล้ว import (แก้ที่นี่ที่เดียว)

In [ ]:
%%writefile t3core.py
import cv2, json, numpy as np

STD_W, STD_H = 1654, 2339          # ขนาดมาตรฐานหลังจัดภาพ (A4 @200dpi)
N_CELLS = 10                        # ช่องต่อแถวคำตอบ
DECIMAL_IDX = 6                     # ช่องที่ 7 (index 6) = จุดทศนิยม (พิมพ์ไว้แล้ว)
HEADER_CELLS = 14                   # ช่องต่อแถวรหัส

def to_std(img):
    return cv2.resize(img, (STD_W, STD_H), interpolation=cv2.INTER_AREA)

def flatten(gray):
    """แก้แสงไม่สม่ำเสมอ: หารด้วยภาพพื้นหลัง (max-filter) -> พื้นกระดาษขาวเท่ากันทั้งภาพ"""
    h, w = gray.shape
    s = 400 / max(h, w)
    small = cv2.resize(gray, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    k = max(3, int(min(small.shape) / 10) | 1)
    bg = cv2.dilate(small, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    bg = cv2.GaussianBlur(bg, (k, k), 0)
    bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return np.clip(gray.astype(np.float32) / (bg + 1) * 255, 0, 255).astype(np.uint8)

def binarize(gray):
    g = cv2.GaussianBlur(gray, (5, 5), 0)
    return cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV, 51, 15)

# ---------------------------------------------------------------- markers
def find_markers(img):
    """หาจุดอ้างอิง 4 จุด: สี่เหลี่ยมดำล่างซ้าย, ล่างขวา, กล่องเลข 3 ขวาบน, ขีดดำแถวแรกซ้าย
    คืนค่า dict {name: (cx, cy)} หรือ None"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    H, W = gray.shape
    fl = flatten(gray)
    otsu, _ = cv2.threshold(fl, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = (fl < min(150, otsu)).astype(np.uint8) * 255   # สี่เหลี่ยมพิมพ์ซีด (เทา ~100-120) ก็ยังทึบ
    k = max(2, round(5 * max(H, W) / STD_H))          # kernel ตามขนาดภาพ (ภาพเล็กใช้ kernel เล็ก กันขีดติดกรอบ)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    page_area = H * W
    squares, dashes = [], []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if w == 0 or h == 0:
            continue
        area = cv2.countNonZero(th[y:y + h, x:x + w])     # นับพิกเซลจริง (แม่นกว่า contourArea ตอนภาพเล็ก)
        ar = w / h
        fill = area / (w * h)
        # สี่เหลี่ยมทึบใหญ่ ~0.25% ของหน้า
        if 0.7 < ar < 1.4 and fill > 0.7 and 0.0008 < area / page_area < 0.01:
            squares.append((x + w / 2, y + h / 2, area))
        # ขีด timing mark (ยาว:สูง ≈ 4:1)
        elif 2.0 < ar < 7 and fill > 0.75 and 0.00005 < area / page_area < 0.001:
            dashes.append((x + w / 2, y + h / 2))
    if len(squares) < 3:
        return None
    # ใช้ 3 สี่เหลี่ยมที่ใหญ่ที่สุด
    squares = sorted(squares, key=lambda s: -s[2])[:3]
    tr = min(squares, key=lambda s: s[1] - s[0])          # บนสุด-ขวาสุด
    rest = [s for s in squares if s is not tr]
    bl, br = sorted(rest, key=lambda s: s[0])
    # ขีดแถวแรกฝั่งซ้าย = ขีดที่ x < กลางหน้า และ y น้อยสุด
    left = [d for d in dashes if d[0] < (bl[0] + br[0]) / 2 and d[1] < bl[1]]
    tl = min(left, key=lambda d: d[1]) if left else None   # ไม่เจอขีด -> ใช้ affine 3 จุด
    return {"tl": tl[:2] if tl else None, "tr": tr[:2], "bl": bl[:2], "br": br[:2]}

def marker_array(m):
    return np.float32([m["tl"], m["tr"], m["br"], m["bl"]])

# ---------------------------------------------------------------- layout (วัดจาก template เปล่า)
def _rects(th, min_w, max_w, ar_lo, ar_hi):
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if min_w <= w <= max_w and ar_lo <= w / h <= ar_hi:
            out.append([x, y, w, h])
    return out

def build_layout(template_img):
    """วัดพิกัดกล่องทั้งหมดจาก template เปล่า (ทำครั้งเดียว) -> dict"""
    img = to_std(template_img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # ดึงเฉพาะเส้นแนวนอน/แนวตั้งยาว ๆ เพื่อหากรอบตาราง
    hl = cv2.morphologyEx(th, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1)))
    vl = cv2.morphologyEx(th, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25)))
    grid = cv2.dilate(hl | vl, np.ones((3, 3), np.uint8))

    # แถวคำตอบ: กว้าง ~ 1/3 หน้า, อัตราส่วน ~9.5
    rows = _rects(grid, STD_W * 0.25, STD_W * 0.45, 7, 13)
    rows = sorted(rows, key=lambda r: (r[0] > STD_W / 2, r[1]))   # คอลัมน์ซ้ายก่อน แล้วบนลงล่าง
    # กล่องรหัส 3 แถว (เฉพาะส่วนช่องกรอก ไม่รวมหัวข้อ): กว้าง ~ 0.48 หน้า อัตราส่วน ~ 13.8
    # กรอบรวม (หัวข้อ+ช่อง) อัตราส่วน ~7 -> เอาครึ่งล่างเป็นแถวช่องกรอก
    hdr = _rects(grid, STD_W * 0.4, STD_W * 0.55, 5, 9.5)
    hdr = [[x, y + h // 2, w, h - h // 2] for x, y, w, h in sorted(hdr, key=lambda r: r[1])]
    # ช่องฝนวงกลม (ชุดคำถาม/หน้า) — วงกลมเล็ก
    cnts, _ = cv2.findContours(th, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    circles = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if 25 <= w <= 45 and 0.8 < w / h < 1.25 and y < STD_H * 0.25 and x < STD_W * 0.4:
            per = cv2.arcLength(c, True)
            if per > 0 and 4 * np.pi * cv2.contourArea(c) / per ** 2 > 0.6:
                circles.append([x, y, w, h])
    circles = _dedup(circles)
    circles = sorted(circles, key=lambda r: r[0])
    layout = {
        "size": [STD_W, STD_H],
        "markers": {k: list(map(float, v)) for k, v in find_markers(img).items()},
        "rows": rows,
        "header": dict(zip(["student_id", "subject_id", "school_id"], hdr)),
        "quiz_set": circles[:3],
        "page_no": circles[3:6],
    }
    return layout

def _dedup(rs, tol=8):
    out = []
    for r in rs:
        if all(abs(r[0] - o[0]) > tol or abs(r[1] - o[1]) > tol for o in out):
            out.append(r)
    return out

# ---------------------------------------------------------------- align
def find_page(img):
    """(ทางเลือก) หาขอบกระดาษ 4 มุม แล้วตัดพื้นหลังออก ถ้าไม่เจอคืนภาพเดิม"""
    ratio = 1000 / max(img.shape[:2])
    small = cv2.resize(img, None, fx=ratio, fy=ratio)
    g = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(g, 50, 150), np.ones((3, 3), np.uint8))
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:5]:
        if cv2.contourArea(c) < 0.3 * small.shape[0] * small.shape[1]:
            break
        approx = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True)
        if len(approx) == 4:
            pts = _order(approx.reshape(4, 2) / ratio)
            M = cv2.getPerspectiveTransform(pts, np.float32([[0, 0], [STD_W, 0], [STD_W, STD_H], [0, STD_H]]))
            return cv2.warpPerspective(img, M, (STD_W, STD_H)), M
    return img, None

def _order(p):
    p = np.float32(p)
    s, d = p.sum(1), np.diff(p, axis=1).ravel()
    return np.float32([p[np.argmin(s)], p[np.argmin(d)], p[np.argmax(s)], p[np.argmax(d)]])

def _align_once(img, layout):
    pre, M0 = find_page(img)
    m = find_markers(pre)
    if m is None and M0 is not None:          # ตัดขอบกระดาษแล้วหาไม่เจอ -> ลองบนภาพเดิม
        pre, M0 = img, None
        m = find_markers(pre)
    if m is None:
        return None, None
    if m["tl"] is not None:
        H = cv2.getPerspectiveTransform(marker_array(m), marker_array(layout["markers"]))
    else:
        k = ["tr", "br", "bl"]
        A = cv2.getAffineTransform(np.float32([m[i] for i in k]), np.float32([layout["markers"][i] for i in k]))
        H = np.vstack([A, [0, 0, 1]])
    aligned = cv2.warpPerspective(pre, H, (STD_W, STD_H), borderValue=(255, 255, 255))
    return aligned, (H @ M0 if M0 is not None else H)

def mask_fields(gray, layout, pad=12):
    """ทาช่องคำตอบ+ช่องรหัสเป็นขาว (ลายมือในช่องไม่ควรมีผลกับ score)"""
    g = gray.copy()
    for x, y, w, h in layout["rows"] + list(layout["header"].values()):
        g[max(0, y - pad):y + h + pad, max(0, x - pad):x + w + pad] = 255
    return g

def match_score(aligned, refs, layout):
    """ความเหมือนกับ template (ใช้ตรวจว่าจัดภาพถูกต้อง/ไม่กลับหัว) — เทียบเฉพาะส่วนที่พิมพ์ไว้
    refs = list ของ template gray (3 หน้า) -> เอาค่าสูงสุด"""
    def vec(g):
        v = cv2.resize(mask_fields(g, layout), (200, 283), interpolation=cv2.INTER_AREA).astype(np.float32)
        return v - v.mean()
    a = vec(flatten(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)))
    return max(float((a * b).sum() / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-6))
               for b in map(vec, refs))

ROTS = [None, cv2.ROTATE_180, cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_90_COUNTERCLOCKWISE]
# วัดจาก 30 รูปจริง + template (2026-10-14): ทิศถูก 0.863–0.999, หมุนผิด/สะท้อน ≤ 0.61 -> ตัดกลาง ๆ
MIN_SCORE = 0.7

def align(img, layout, refs, min_score=None):
    """จัดภาพถ่ายให้ตรง template ลองหมุน 0/180/90/270 แล้วเลือกที่เหมือน template ที่สุด
    คืน (aligned_img, H, rot, score) — H แมปจากภาพที่หมุนแล้ว -> ภาพมาตรฐาน"""
    min_score = MIN_SCORE if min_score is None else min_score
    best = (None, None, None, -1)
    for rot in ROTS:
        im = img if rot is None else cv2.rotate(img, rot)
        al, H = _align_once(im, layout)
        if al is None:
            continue
        sc = match_score(al, refs, layout)
        if sc > best[3]:
            best = (al, H, rot, sc)
        if sc > 0.8:
            break
    if best[0] is None or best[3] < min_score:
        raise RuntimeError(f"จัดภาพไม่สำเร็จ (score={best[3]:.2f}) — ตรวจว่าเห็นสี่เหลี่ยมดำครบ 3 มุม")
    return best

# ---------------------------------------------------------------- cells
def refine_rect(gray, rect, search=15):
    """ขยับขอบกรอบทั้ง 4 ด้านให้ทับเส้นขอบจริงของกล่อง (หาแยกซ้าย/ขวา/บน/ล่าง)
    แก้ความคลาดเคลื่อนจากกระดาษโค้ง ภาพเล็ก หรือขนาดพิมพ์ต่างกันเล็กน้อย"""
    x, y, w, h = map(int, rect)
    s = search
    H, W = gray.shape
    if x - s < 0 or y - s < 0 or x + w + s >= W or y + h + s >= H:
        return [x, y, w, h]
    dark = (gray[y - s:y + h + s + 1, x - s:x + w + s + 1] < 150).astype(np.float32)
    k = np.ones(3) / 3

    def edge(profile, nominal):
        seg = np.convolve(profile, k, 'same')[nominal - s:nominal + s + 1]
        d = int(np.argmax(seg - 0.002 * np.abs(np.arange(-s, s + 1))))   # เท่ากัน -> เลือกที่ใกล้เดิม
        return d - s if seg.max() > 0.5 else 0                            # ไม่เจอเส้นชัด -> ไม่ขยับ

    rp = dark[:, s + 5:s + w - 5].mean(1)                 # เส้นแนวนอน (บน/ล่าง)
    t, bt = edge(rp, s), edge(rp, s + h)
    cp = dark[s + t + 5:s + h + bt - 5, :].mean(0)        # เส้นแนวตั้ง (ซ้าย/ขวา) ไม่นับเส้นบนล่าง
    l, r = edge(cp, s), edge(cp, s + w)
    return [x + l, y + t, w + r - l, h + bt - t]

def split_row(aligned, rect, n=N_CELLS, pad=0.14, refine=True):
    """แบ่งกรอบแถวเป็น n ช่องเท่ากัน ตัดขอบเส้นออก pad% -> list ของภาพ grayscale"""
    x, y, w, h = rect
    # ส่งภาพ gray ที่ flatten แล้วมาจะเร็วกว่า (ทำครั้งเดียวต่อหน้า)
    gray = flatten(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)) if aligned.ndim == 3 else aligned
    if refine:
        x, y, w, h = refine_rect(gray, rect)
    cw = w / n
    cells = []
    for i in range(n):
        x0, x1 = x + i * cw, x + (i + 1) * cw
        px, py = cw * pad, h * pad
        cells.append(gray[int(y + py):int(y + h - py), int(x0 + px):int(x1 - px)])
    return cells

INK_LEVEL = 185   # ค่าความสว่าง (หลัง flatten) ที่ถือว่าเป็นหมึกสำหรับ Character Check — กระดาษ ~255

DIGIT_LEVEL = 200  # threshold สำหรับตัดรูปตัวเลข (หลวมกว่า เก็บเส้นจาง)

def ink_mask(cell, level=None):
    m = (cell < (level or INK_LEVEL)).astype(np.uint8) * 255
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))   # ลบจุด noise

def remove_border_lines(th):
    """ลบเศษเส้นตารางที่ติดขอบช่อง (เส้นยาวบาง แนวตั้ง/แนวนอน ที่แตะขอบ)"""
    h, w = th.shape
    n, lab, st, _ = cv2.connectedComponentsWithStats(th)
    for i in range(1, n):
        x, y, bw, bh = st[i, :4]
        vline = bh > 0.5 * h and bw < 0.15 * w and (x == 0 or x + bw >= w)          # ต้องแตะขอบจริง (กันลบเลข 1)
        hline = bw > 0.5 * w and bh < 0.15 * h and (y <= 0.15 * h or y + bh >= 0.85 * h)
        if vline or hline:
            th[lab == i] = 0
    return th

def ink_ratio(cell):
    """สัดส่วนหมึกในช่อง (Character Check: ว่างหรือไม่) ไม่นับเศษเส้นตาราง"""
    return float(remove_border_lines(ink_mask(cell)).mean() / 255)

def bubble_filled(aligned, circles):
    gray = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)
    vals = [255 - gray[y + h // 4:y + 3 * h // 4, x + w // 4:x + 3 * w // 4].mean() for x, y, w, h in circles]
    if max(vals) < 80:                        # ไม่มีวงไหนถูกฝน
        return 0, vals
    return int(np.argmax(vals)) + 1, vals

def prep_cell(cell, size=32):
    """ทำให้ช่องเป็นภาพขาวดำ 32x32 ตัวเลขอยู่กลาง (แบบ MNIST)"""
    th = ink_mask(cell, level=DIGIT_LEVEL)          # threshold หลวมกว่า -> เส้นปากกาจาง ๆ ไม่ขาด
    th = remove_border_lines(th)
    # เก็บเฉพาะ component ที่ใหญ่พอ (ตัดเศษเส้นตาราง/จุด)
    n, lab, st, _ = cv2.connectedComponentsWithStats(th)
    keep = [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= 0.15 * st[1:, cv2.CC_STAT_AREA].max()] if n > 1 else []
    th = np.isin(lab, keep).astype(np.uint8) * 255
    ys, xs = np.nonzero(th)
    if len(xs) < 15:
        return np.zeros((size, size), np.uint8)
    d = th[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    s = max(d.shape) + 8
    canvas = np.zeros((s, s), np.uint8)
    oy, ox = (s - d.shape[0]) // 2, (s - d.shape[1]) // 2
    canvas[oy:oy + d.shape[0], ox:ox + d.shape[1]] = d
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)

In [ ]:
import cv2, json, glob, random, numpy as np, matplotlib.pyplot as plt
import importlib, t3core as core; importlib.reload(core)

def show(img, title='', size=8):
    plt.figure(figsize=(size, size * 1.4))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap='gray')
    plt.title(title); plt.axis('off'); plt.show()

IMG_EXT = ('.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp')
def list_imgs(folder):
    # รูปทุกไฟล์ในโฟลเดอร์ (รวมโฟลเดอร์ย่อย)
    return sorted(f for f in glob.glob(f'{folder}/**/*', recursive=True) if f.lower().endswith(IMG_EXT))

def imread(path):
    # อ่านรูปได้ทั้ง jpg/png และ HEIC จากไอโฟน + หมุนตาม EXIF
    from PIL import Image, ImageOps
    if path.lower().endswith(('.heic', '.heif')):
        import pillow_heif; pillow_heif.register_heif_opener()
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR)

## 2. วัด layout จาก template เปล่า (ทำครั้งเดียว)
ได้พิกัด 30 แถวคำตอบ, แถวรหัส 3 แถว, วงกลมชุดคำถาม/หน้า และ marker 4 จุด → เก็บเป็น `layout_p1..3.json`

In [ ]:
LAYOUT, REF = {}, {}
for p in [1, 2, 3]:
    f = [x for x in list_imgs(TEMPLATE_DIR) if f'page{p}' in os.path.basename(x)][0]
    tpl = imread(f)
    L = core.build_layout(tpl)
    assert len(L['rows']) == 30 and len(L['header']) == 3, f'page{p}: หา layout ไม่ครบ'
    LAYOUT[p] = L
    REF[p] = cv2.cvtColor(core.to_std(tpl), cv2.COLOR_BGR2GRAY)
    json.dump(L, open(f'{BASE}/out/layout_p{p}.json', 'w'))
    print(f'page{p}: rows={len(L["rows"])} header={list(L["header"])} page-bubble={core.bubble_filled(core.to_std(tpl), L["page_no"])[0]}')

def draw_layout(img, L):
    a = img.copy()
    for x, y, w, h in L['rows']:
        cv2.rectangle(a, (x, y), (x + w, y + h), (0, 0, 255), 3)
        for k in range(1, 10):
            xx = int(x + k * w / 10); cv2.line(a, (xx, y), (xx, y + h), (255, 0, 0), 1)
    for x, y, w, h in L['header'].values():
        cv2.rectangle(a, (x, y), (x + w, y + h), (0, 160, 0), 3)
        for k in range(1, 14):
            xx = int(x + k * w / 14); cv2.line(a, (xx, y), (xx, y + h), (255, 0, 0), 1)
    for x, y, w, h in L['quiz_set'] + L['page_no']:
        cv2.rectangle(a, (x, y), (x + w, y + h), (255, 0, 255), 3)
    for v in L['markers'].values():
        cv2.circle(a, tuple(map(int, v)), 14, (0, 200, 255), -1)
    return a

show(draw_layout(core.to_std(imread([x for x in list_imgs(TEMPLATE_DIR) if 'page1' in x][0])), LAYOUT[1]), 'layout page 1')

## 3. ทดสอบจัดภาพกับรูปถ่ายจริง 1 รูป
ลองหมุน 0/180/90/270 อัตโนมัติ แล้วเลือกที่ตรงกับ template มากที่สุด (score ควร > 0.6)

In [ ]:
def process_sheet(path):
    """รูปถ่าย -> (aligned, page_no, layout, H, rot)"""
    img = imread(path)
    aligned, H, rot, score = core.align(img, LAYOUT[1], [REF[p] for p in (1, 2, 3)])
    page, _ = core.bubble_filled(aligned, LAYOUT[1]['page_no'])
    return aligned, page, LAYOUT[page], H, rot, score

test_path = (list_imgs(f'{RAW_DIR}/0') + list_imgs(f'{BASE}/eval') + list_imgs(TEMPLATE_DIR))[0]
aligned, page, L, H, rot, score = process_sheet(test_path)
print(os.path.basename(test_path), '| page', page, '| rotate', rot, '| score', round(score, 3))
show(draw_layout(aligned, L), 'aligned + grid')

g = core.flatten(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY))
fig, ax = plt.subplots(3, 10, figsize=(12, 4))
for r in range(3):
    for i, c in enumerate(core.split_row(g, L['rows'][r])):
        ax[r, i].imshow(c, cmap='gray'); ax[r, i].axis('off')
        ax[r, i].set_title(f'{core.ink_ratio(c):.2f}', fontsize=7)
plt.suptitle('ข้อ 1-3 แบ่ง 10 ช่อง (ตัวเลข = ink ratio)'); plt.show()

## 4. สร้าง dataset ตัวเลขสำหรับ CNN
**วิธีเก็บข้อมูล:** แต่ละแผ่นให้คนเขียน **เลขเดียวกันทุกช่อง** แล้วเก็บรูปในโฟลเดอร์ `Raw_Dataset/<เลข>/` → label ได้อัตโนมัติ
- ช่องที่ ink ratio ต่ำ = คลาส `blank` (ช่องที่ไม่ได้เขียน)
- แบ่ง train/val/test **ตามแผ่น** (ไม่ใช่ตามช่อง) กันข้อมูลรั่ว เพราะลายมือแผ่นเดียวกันจะคล้ายกันมาก
- ข้ามช่องที่ 7 (จุดทศนิยมพิมพ์ไว้แล้ว)

In [ ]:
import csv
from collections import Counter
INK_TH = 0.04           # ปรับได้จาก histogram ด้านล่าง
CELL_DIR = f'{BASE}/out/cells'
CLASSES = [str(i) for i in range(10)] + ['blank']

def sheet_cells(aligned, L):
    """คืน list ของ (field, q, idx, cell_gray) ทุกช่องในหน้า"""
    g = core.flatten(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY))
    out = []
    for q, r in enumerate(L['rows']):
        for i, c in enumerate(core.split_row(g, r)):
            if i != core.DECIMAL_IDX:
                out.append(('ans', q, i, c))
    for name, r in L['header'].items():
        for i, c in enumerate(core.split_row(g, r, n=core.HEADER_CELLS)):
            out.append((name, 0, i, c))
    return out

# แบ่ง train/val/test
#  - คลาสที่มี >= 3 แผ่น: แบ่งตามแผ่น 70/15/15 (ถูกต้องที่สุด ไม่มีลายมือรั่วข้าม split)
#  - คลาสที่มี < 3 แผ่น (ช่วงทดลอง): แบ่งตาม "ข้อ" ในแผ่น -> ข้อ 1-20 train, 21-25 val, 26-30 test
#    (ใช้ทดสอบ pipeline ได้ แต่ accuracy จะสูงเกินจริงเพราะลายมือเดียวกัน — รายงานจริงต้องเก็บเพิ่ม)
import shutil
shutil.rmtree(CELL_DIR, ignore_errors=True)
random.seed(42)
sheets = []
for lab in CLASSES:
    fs = list_imgs(f'{RAW_DIR}/{lab}'); random.shuffle(fs)
    n = len(fs)
    n_te = max(1, round(0.15 * n))
    n_va = max(1, round(0.15 * n))
    for j, f in enumerate(fs):
        if n < 3:
            split = 'by_row'
        else:
            split = 'test' if j < n_te else ('val' if j < n_te + n_va else 'train')
        sheets.append((f, lab, split))
print('จำนวนแผ่น:', dict(Counter(s[1] for s in sheets)))
print('วิธีแบ่ง  :', dict(Counter(s[2] for s in sheets)))

def row_split(field, q):
    if field != 'ans': return 'train'                     # ช่องรหัส header -> train
    q = q % 30
    return 'train' if q < 20 else ('val' if q < 25 else 'test')

rows, inks, fails = [], [], []
for f, lab, split in sheets:
    try:
        aligned, page, L, *_ = process_sheet(f)
    except Exception as e:
        fails.append((f, str(e))); continue
    sid = os.path.splitext(os.path.basename(f))[0]
    for field, q, i, c in sheet_cells(aligned, L):
        ink = core.ink_ratio(c); inks.append((ink, lab))
        y = 'blank' if (lab == 'blank' or ink < INK_TH) else lab
        sp = row_split(field, q) if split == 'by_row' else split
        d = f'{CELL_DIR}/{sp}/{y}'; os.makedirs(d, exist_ok=True)
        fn = f'{d}/{lab}_{sid}_p{page}_{field}_q{q+1:02d}_c{i}.png'
        cv2.imwrite(fn, core.prep_cell(c, 32))
        rows.append([fn, y, sid, sp])

with open(f'{BASE}/out/cells.csv', 'w', newline='') as fp:
    csv.writer(fp).writerows([['path', 'label', 'sheet', 'split']] + rows)
print('ช่องทั้งหมด', len(rows), '| แผ่นที่จัดภาพไม่ได้', len(fails)); [print('  ', x) for x in fails]
print(Counter((r[3], r[1]) for r in rows))

# histogram ink ratio: แผ่นเปล่า vs แผ่นที่เขียน -> ใช้เลือก INK_TH
a = [i for i, l in inks if l == 'blank']; b = [i for i, l in inks if l != 'blank']
plt.hist(b, 80, alpha=.6, label='แผ่นเขียน'); plt.hist(a, 80, alpha=.6, label='แผ่นเปล่า')
plt.axvline(INK_TH, c='r'); plt.yscale('log'); plt.legend(); plt.title('ink ratio'); plt.show()

In [ ]:
# ตรวจตาด้วยตัวเอง: สุ่มดูช่องละ 12 รูปต่อคลาส (หา label ผิด เช่นคนเขียนผิดเลข)
fig, ax = plt.subplots(len(CLASSES), 12, figsize=(12, 11))
for r, lab in enumerate(CLASSES):
    fs = glob.glob(f'{CELL_DIR}/train/{lab}/*.png')
    for c in range(12):
        ax[r, c].axis('off')
        if c < len(fs):
            ax[r, c].imshow(cv2.imread(random.choice(fs), 0), cmap='gray')
    ax[r, 0].set_title(lab, fontsize=9, loc='left')
plt.tight_layout(); plt.show()

## 5. YOLO: หาแถวคำตอบ + ช่องรหัส
**Label อัตโนมัติ** จาก layout (ไม่ต้องวาดกรอบเอง) บนภาพที่จัดแล้ว
คลาส: `answer_row`, `student_id`, `subject_id`, `school_id`, `quiz_set`, `page_no`

เปิด augmentation (หมุน/เลื่อน/ย่อขยาย/perspective) เพื่อให้ YOLO ทนต่อภาพที่จัดไม่เป๊ะ — **ปิด flip** เพราะเอกสารกลับซ้ายขวาไม่ได้

In [ ]:
YOLO_DIR = '/content/yolo_t3'          # เก็บในเครื่อง Colab (เร็วกว่า Drive)
YOLO_CLASSES = ['answer_row', 'student_id', 'subject_id', 'school_id', 'quiz_set', 'page_no']
IMG_W = 960                             # ย่อภาพสำหรับ YOLO
scale = IMG_W / core.STD_W

def union(rs):
    xs = [r[0] for r in rs]; ys = [r[1] for r in rs]
    return [min(xs), min(ys), max(r[0] + r[2] for r in rs) - min(xs), max(r[1] + r[3] for r in rs) - min(ys)]

def yolo_labels(L, pad=6):
    objs = [(0, r) for r in L['rows']]
    objs += [(YOLO_CLASSES.index(k), r) for k, r in L['header'].items()]
    objs += [(4, union(L['quiz_set'])), (5, union(L['page_no']))]
    lines = []
    for c, (x, y, w, h) in objs:
        x, y, w, h = x - pad, y - pad, w + 2 * pad, h + 2 * pad
        lines.append(f'{c} {(x + w/2)/core.STD_W:.6f} {(y + h/2)/core.STD_H:.6f} {w/core.STD_W:.6f} {h/core.STD_H:.6f}')
    return '\n'.join(lines)

# YOLO ใช้แค่ train/val — ถ้าแผ่นน้อย (by_row) สุ่ม 20% ไปเป็น val
for f, lab, split in sheets:
    sp = {'test': 'val', 'by_row': 'val' if random.random() < 0.2 else 'train'}.get(split, split)
    try:
        aligned, page, L, *_ = process_sheet(f)
    except Exception:
        continue
    sid = f'{lab}_{os.path.splitext(os.path.basename(f))[0]}'
    for d in ['images', 'labels']:
        os.makedirs(f'{YOLO_DIR}/{d}/{sp}', exist_ok=True)
    cv2.imwrite(f'{YOLO_DIR}/images/{sp}/{sid}.jpg', cv2.resize(aligned, None, fx=scale, fy=scale))
    open(f'{YOLO_DIR}/labels/{sp}/{sid}.txt', 'w').write(yolo_labels(L))

open(f'{YOLO_DIR}/data.yaml', 'w').write(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnames: {YOLO_CLASSES}\n')
print({sp: len(os.listdir(f'{YOLO_DIR}/images/{sp}')) for sp in ['train', 'val']})

In [ ]:
from ultralytics import YOLO
yolo = YOLO('yolov8n.pt')
yolo.train(data=f'{YOLO_DIR}/data.yaml', imgsz=IMG_W, epochs=60, batch=16, patience=15,
           fliplr=0.0, mosaic=0.5, degrees=3, translate=0.05, scale=0.2, perspective=0.0005,
           hsv_v=0.4, project=f'{BASE}/out/yolo', name='t3', exist_ok=True)

In [ ]:
yolo = YOLO(f'{BASE}/out/yolo/t3/weights/best.pt')
m = yolo.val(data=f'{YOLO_DIR}/data.yaml', imgsz=IMG_W)
print('mAP50 =', round(m.box.map50, 4), '| mAP50-95 =', round(m.box.map, 4))
val_img = glob.glob(f'{YOLO_DIR}/images/val/*.jpg')[0]
show(yolo(val_img, imgsz=IMG_W, verbose=False)[0].plot(line_width=1), 'YOLO prediction')

## 6. CNN ที่ propose: **DigitNet-T3**
- Input 32×32 ขาวดำ (ตัวเลขขาวพื้นดำ จัดกึ่งกลางแบบ MNIST)
- 3 บล็อก Conv-BN-ReLU ×2 + MaxPool → Global Average Pooling → Dropout → FC 11 คลาส (0–9, blank)
- ~0.29M พารามิเตอร์ เล็กพอรันเร็วบน CPU
- (ทางเลือก) **pre-train ด้วย MNIST** แล้ว fine-tune ด้วยข้อมูลจริง ช่วยมากตอนข้อมูลน้อย

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

class DigitNet(nn.Module):
    def __init__(self, n_cls=11):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1, 32), block(32, 64), block(64, 128))   # 32->16->8->4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_cls))
    def forward(self, x):
        return self.head(self.features(x))

print('params:', sum(p.numel() for p in DigitNet().parameters()))

aug = T.Compose([T.ToPILImage(), T.RandomAffine(8, translate=(0.08, 0.08), scale=(0.85, 1.1), shear=8),
                 T.ToTensor(), T.Normalize([0.13], [0.3])])
plain = T.Compose([T.ToPILImage(), T.ToTensor(), T.Normalize([0.13], [0.3])])

class CellDS(Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.path.tolist(), [CLASSES.index(l) for l in df.label], tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(cv2.imread(self.p[i], 0)), self.y[i]

df = pd.read_csv(f'{BASE}/out/cells.csv')
dl = {s: DataLoader(CellDS(df[df.split == s], aug if s == 'train' else plain), batch_size=128,
                    shuffle=(s == 'train'), num_workers=2) for s in ['train', 'val', 'test']}
print(df.groupby(['split']).size())

In [ ]:
def run_epoch(model, loader, opt=None):
    model.train(opt is not None); tot = correct = loss_sum = 0
    with torch.set_grad_enabled(opt is not None):
        for x, y in loader:
            x, y = x.to(dev), y.to(dev)
            out = model(x); loss = F.cross_entropy(out, y, label_smoothing=0.05)
            if opt: opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); tot += len(y)
    return loss_sum / tot, correct / tot

def fit(model, train_dl, val_dl, epochs, lr, tag):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * len(train_dl))
    best, hist = 0, []
    for ep in range(epochs):
        model.train(); tl = ta = n = 0
        for x, y in train_dl:
            x, y = x.to(dev), y.to(dev)
            out = model(x); loss = F.cross_entropy(out, y, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
            tl += loss.item() * len(y); ta += (out.argmax(1) == y).sum().item(); n += len(y)
        vl, va = run_epoch(model, val_dl)
        hist.append((tl / n, ta / n, vl, va))
        if va >= best: best = va; torch.save(model.state_dict(), f'{BASE}/out/{tag}.pt')
        print(f'[{tag}] ep{ep+1:02d} train {ta/n:.4f}  val {va:.4f}')
    model.load_state_dict(torch.load(f'{BASE}/out/{tag}.pt'))
    return hist

USE_MNIST_PRETRAIN = True
model = DigitNet().to(dev)
if USE_MNIST_PRETRAIN:
    # MNIST 0-9 (ไม่มี blank) -> pad เป็น 32x32 ให้เหมือน prep_cell
    mt = T.Compose([T.Pad(2), T.RandomAffine(8, translate=(0.08, 0.08), scale=(0.85, 1.1)), T.ToTensor(), T.Normalize([0.13], [0.3])])
    mv = T.Compose([T.Pad(2), T.ToTensor(), T.Normalize([0.13], [0.3])])
    m_tr = DataLoader(datasets.MNIST('/content/mnist', True, mt, download=True), 256, shuffle=True, num_workers=2)
    m_va = DataLoader(datasets.MNIST('/content/mnist', False, mv, download=True), 512, num_workers=2)
    fit(model, m_tr, m_va, epochs=3, lr=3e-3, tag='digitnet_mnist')

hist = fit(model, dl['train'], dl['val'], epochs=30, lr=2e-3, tag='digitnet_t3')
h = np.array(hist)
plt.plot(h[:, 1], label='train acc'); plt.plot(h[:, 3], label='val acc'); plt.legend(); plt.grid(); plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
model.eval(); P, Y = [], []
with torch.no_grad():
    for x, y in dl['test']:
        P += model(x.to(dev)).argmax(1).cpu().tolist(); Y += y.tolist()
print(classification_report(Y, P, labels=range(11), target_names=CLASSES, digits=4, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(Y, P, labels=range(11)), display_labels=CLASSES).plot(cmap='Blues')
plt.title('DigitNet-T3 — test set (แผ่นที่ไม่เคยเห็น)'); plt.show()

## 7. ตรวจข้อสอบจริง → JSON → คะแนน
1. จัดภาพ → อ่านหน้า/ชุดคำถามจากวงกลม
2. **YOLO** หาแถวคำตอบ → จับคู่กับข้อด้วยตำแหน่ง (ถ้า YOLO พลาดแถวไหนจะใช้พิกัด layout แทน)
3. แบ่ง 10 ช่อง → Character Check → DigitNet → รวมเป็นคำตอบ เช่น `12.5`

**ไฟล์เฉลยสำหรับประเมิน** `eval/IMG_xxx.csv`:
```
field,value
student_id,65123456
q1,12.5
q2,3
...
```

In [ ]:
CONF_TH = 0.6      # ถ้าความมั่นใจของ CNN ต่ำกว่านี้ ติดธงให้คนตรวจซ้ำ

@torch.no_grad()
def read_cells(cells):
    """cells: list ภาพ gray -> list ของ (ตัวอักษร, ความมั่นใจ)"""
    model.eval(); out, batch, idx = [], [], []
    for k, c in enumerate(cells):
        if core.ink_ratio(c) < INK_TH:
            out.append(('', 1.0))                               # Character Check: ว่าง
        else:
            out.append(None); idx.append(k); batch.append(plain(core.prep_cell(c, 32)))
    if batch:
        pr = F.softmax(model(torch.stack(batch).to(dev)), 1).cpu()
        for k, p in zip(idx, pr):
            j = int(p.argmax()); out[k] = ('' if CLASSES[j] == 'blank' else CLASSES[j], float(p[j]))
    return out

def yolo_boxes(aligned):
    r = yolo(cv2.resize(aligned, None, fx=scale, fy=scale), imgsz=IMG_W, conf=0.4, verbose=False)[0]
    return [(int(c), (b / scale).tolist()) for c, b in zip(r.boxes.cls.cpu(), r.boxes.xyxy.cpu())]

def match(target, boxes, cls, tol=40):
    """หา box ของ YOLO ที่ตรงกับกรอบ layout (x,y,w,h) ถ้าไม่เจอคืนกรอบ layout เดิม"""
    tx, ty = target[0] + target[2] / 2, target[1] + target[3] / 2
    best = None
    for c, (x1, y1, x2, y2) in boxes:
        d = abs((x1 + x2) / 2 - tx) + abs((y1 + y2) / 2 - ty)
        if c == cls and d < tol and (best is None or d < best[0]):
            best = (d, [x1 + 6, y1 + 6, x2 - x1 - 12, y2 - y1 - 12])  # ตัด pad 6px ที่ใส่ตอน label
    return (best[1], True) if best else (target, False)

def read_sheet(path):
    aligned, page, L, H, rot, score = process_sheet(path)
    g = core.flatten(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY))
    boxes = yolo_boxes(aligned)
    res = {'file': os.path.basename(path), 'page': page,
           'quiz_set': core.bubble_filled(aligned, L['quiz_set'])[0], 'answers': {}, 'flags': []}
    for name, r in L['header'].items():
        rect, _ = match(r, boxes, YOLO_CLASSES.index(name))
        chars = read_cells(core.split_row(g, rect, n=core.HEADER_CELLS))
        res[name] = ''.join(c for c, _ in chars)
    n_yolo = 0
    for q, r in enumerate(L['rows']):
        rect, hit = match(r, boxes, 0); n_yolo += hit
        cells = core.split_row(g, rect)
        chars = read_cells(cells[:core.DECIMAL_IDX] + cells[core.DECIMAL_IDX + 1:])   # ข้ามช่องจุดทศนิยม
        intp = ''.join(c for c, _ in chars[:core.DECIMAL_IDX])
        decp = ''.join(c for c, _ in chars[core.DECIMAL_IDX:])
        qn = (page - 1) * 30 + q + 1
        res['answers'][qn] = (intp or ('0' if decp else '')) + ('.' + decp if decp else '')
        if min(p for _, p in chars) < CONF_TH:
            res['flags'].append(qn)
    res['yolo_rows_found'] = n_yolo
    return res

def score_sheet(res, key):
    """key: {qn: 'คำตอบ'} เทียบเป็นตัวเลข (12.50 == 12.5)"""
    s = 0
    for qn, a in key.items():
        try: s += abs(float(res['answers'].get(qn, 'nan')) - float(a)) < 1e-9
        except ValueError: pass
    return s, len(key)

r = read_sheet(test_path)
print(json.dumps(r, ensure_ascii=False, indent=1)[:1500])

In [ ]:
# ประเมิน CAR / WAR บนแผ่นทดสอบจริงที่มีเฉลย (eval/*.jpg + *.csv)
def norm(v):
    return '' if pd.isna(v) else str(v).strip()

def car_war(pairs):
    """CAR = ตัวอักษรถูก / ตัวอักษรทั้งหมด (edit distance), WAR = ฟิลด์ถูกทั้งฟิลด์ / ฟิลด์ทั้งหมด"""
    import difflib
    ch_ok = ch_all = w_ok = 0
    for gt, pr in pairs:
        sm = difflib.SequenceMatcher(None, gt, pr)
        ch_ok += sum(b.size for b in sm.get_matching_blocks()); ch_all += max(len(gt), len(pr), 1)
        w_ok += gt == pr
    return ch_ok / ch_all, w_ok / len(pairs)

pairs, all_res = [], []
for f in list_imgs(f'{BASE}/eval'):
    k = os.path.splitext(f)[0] + '.csv'
    if not os.path.exists(k): continue
    gt = {row.field: norm(row.value) for row in pd.read_csv(k, dtype=str).itertuples()}
    r = read_sheet(f); all_res.append(r)
    for fld, v in gt.items():
        pr = r['answers'].get(int(fld[1:]), '') if fld.startswith('q') else r.get(fld, '')
        pairs.append((v, pr))
        if v != pr: print(f'  ✗ {r["file"]} {fld}: เฉลย={v!r} อ่านได้={pr!r}')
if pairs:
    car, war = car_war(pairs)
    print(f'\nCAR = {car:.4f}   WAR = {war:.4f}   (baseline T3: CAR 0.9880, WAR 0.9844)')
json.dump(all_res, open(f'{BASE}/out/results.json', 'w'), ensure_ascii=False, indent=1)